# 02 - Convolutional VAE

Learns a 16-dimensional summary of each daily scalogram (128 scales x 24 hours). A Gaussian mixture on those summaries assigns each day to a market regime.

Pipeline: scalogram -> ConvVAE encoder -> latent mean (mu) -> GMM -> regime

Methodology notes:
1. Days are split chronologically (60% train, 20% validation, 20% test). The 2023-2024 El Niño falls in the test period.
2. The GMM is fitted on training days only and then applied to all days.

<d.benavidess@uniandes.edu.co>

In [ ]:
import sys
sys.path.append("..")

import json
from pathlib import Path

import torch

import numpy as np
import pandas as pd
import altair as alt
import matplotlib.pyplot as plt
from prettytable import PrettyTable
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

plt.rcParams['font.family'] = 'Arial'
alt.data_transformers.enable('vegafusion')

from src.data import load_scalograms, load_context, chronological_split, ScalogramDataset
from src.models.conv_vae import ConvVAE
from src.training import set_seed, train, save_run
from src.clustering import assign_regimes, summary_metrics, regime_profile, save_regimes
from src.plots import loss_curves, selection_chart, tsne_chart, regime_timeline, plot_reconstructions

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## 0. Experiment Run

In [ ]:
# Update the run name to reflect the current experiment
RUN_NAME = "run01_k8"

## 1. Configuration

In [ ]:
SCALOGRAM_DIR = Path("../data/processed/scalograms")
DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../models/vae") / RUN_NAME
FIGURE_DIR = Path("../figures/vae") / RUN_NAME
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "z_dim": 16,
    "filters": [32, 64, 128],
    "beta": 1.0,
    "kl_warmup_epochs": 0,
    "lr": 1e-3,
    "batch_size": 64,
    "epochs": 30,
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "n_regimes": 8,
    "k_range": [2, 3, 4, 5, 6, 7, 8],
    "seed": 42,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(CONFIG["seed"])
print(f"Device: {DEVICE}")

## 2. Data

In [ ]:
images, metadata = load_scalograms(SCALOGRAM_DIR)
context = load_context(DATA_DIR, metadata)
train_idx, val_idx, test_idx = chronological_split(len(images), CONFIG["train_ratio"], CONFIG["val_ratio"])
split_dates = [context["date"].iloc[val_idx[0]], context["date"].iloc[test_idx[0]]]

IMG_C, IMG_H, IMG_W = images.shape[1:]
print(f"Scalograms: {images.shape}, range [{images.min():.3f}, {images.max():.3f}]")

t = PrettyTable(["Split", "Days", "From", "To"])
for name, idx in (("train", train_idx), ("val", val_idx), ("test", test_idx)):
    t.add_row([name, len(idx), context["date"].iloc[idx[0]].date(), context["date"].iloc[idx[-1]].date()])
print(t)

In [ ]:
train_loader = DataLoader(ScalogramDataset(images, train_idx), batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(ScalogramDataset(images, val_idx), batch_size=CONFIG["batch_size"], shuffle=False)

## 3. Model

In [ ]:
model = ConvVAE(CONFIG["z_dim"], IMG_C, IMG_H, IMG_W, CONFIG["filters"]).to(DEVICE)

with torch.no_grad():
    out_shape = tuple(model(torch.from_numpy(images[:2]).to(DEVICE))[0].shape[1:])
assert out_shape == (IMG_C, IMG_H, IMG_W), f"decoder output {out_shape} does not match input"

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Encoder feature map: {model.encoder.feat_shape}")
print(f"Decoder output: {out_shape}")

## 4. Training

In [ ]:
history = train(model, train_loader, val_loader, DEVICE, epochs=CONFIG["epochs"], lr=CONFIG["lr"],
                beta=CONFIG["beta"], kl_warmup_epochs=CONFIG["kl_warmup_epochs"])
save_run(MODEL_DIR, model, CONFIG, history)

## 5. Loss curves

Posterior collapse check: if the KL curve drops close to zero, the encoder is ignoring the input. In that case try `beta` below 1 or set `kl_warmup_epochs` (for example 10). Active dimensions counts the latent dimensions whose mean actually varies across days.

In [ ]:
loss_plot = loss_curves(history, components=("total", "recon", "kl"))
loss_plot

In [ ]:
loss_plot.save(str(FIGURE_DIR / "loss_curves.png"), scale_factor=2.0)

## 6. Reconstructions

Test-period days.

In [ ]:
model.eval()
show_idx = test_idx[np.linspace(0, len(test_idx) - 1, 8).astype(int)]
with torch.no_grad():
    recon = model(torch.from_numpy(images[show_idx]).to(DEVICE))[0].cpu().numpy()

fig = plot_reconstructions(images[show_idx], recon, [str(d.date()) for d in context["date"].iloc[show_idx]])
fig.savefig(FIGURE_DIR / "reconstructions.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Latent vectors

In [ ]:
latents = model.extract_latents(images, DEVICE)

np.save(MODEL_DIR / "latents.npy", latents)

active = int((latents.var(axis=0) > 0.01).sum())
print(f"Latent vectors: {latents.shape}")
print(f"Final train KL: {history[-1]['train_kl']:.2f}")
print(f"Active dimensions: {active} of {CONFIG['z_dim']}")

## 8. Regimes with a Gaussian mixture

The GMM is fitted on the training days only and then assigns every day. The number of regimes is the one with the lowest BIC unless `CONFIG["n_regimes"]` fixes it. Regimes are numbered by ascending mean real price, so regime 0 is always the cheapest and labels are comparable across models.

In [ ]:
selection, K, labels, probs = assign_regimes(
    latents, train_idx, context["precio_real_cop_kwh"].to_numpy(),
    k=CONFIG["n_regimes"], k_range=CONFIG["k_range"], seed=CONFIG["seed"],
)
context["regime"] = labels

t = PrettyTable(["k", "BIC", "Silhouette"])
for _, row in selection.iterrows():
    t.add_row([int(row["k"]), f"{row['bic']:,.0f}", f"{row['silhouette']:.3f}"])
print(t)
print(f"Selected k = {K}")
if CONFIG["n_regimes"] is None and K == max(CONFIG["k_range"]):
    print("BIC is still falling at the largest k. Choose k from the table and set CONFIG['n_regimes'] by hand.")

In [ ]:
selection_plot = selection_chart(selection, K)
selection_plot

In [ ]:
selection_plot.save(str(FIGURE_DIR / "gmm_selection.png"), scale_factor=2.0)

## 9. Latent space

In [ ]:
rng = np.random.default_rng(CONFIG["seed"])
plot_idx = np.sort(rng.choice(len(latents), size=min(5000, len(latents)), replace=False))
coords = TSNE(n_components=2, perplexity=30, random_state=CONFIG["seed"]).fit_transform(latents[plot_idx])

tsne_plot = tsne_chart(coords, labels[plot_idx], f"t-SNE of the latent space (k={K})")
tsne_plot

In [ ]:
tsne_plot.save(str(FIGURE_DIR / "tsne_regimes.png"), scale_factor=2.0)

## 10. Regime timeline

Shaded bands are El Nino months (ONI >= 0.5). Dashed lines mark the start of the validation and test periods. First sanity check: the high-price regimes should concentrate in the 2015-2016 and 2023-2024 El Nino episodes and in low reservoir periods.

In [ ]:
timeline_plot = regime_timeline(context, split_dates, title=f"Detected regimes over time (k={K})")
timeline_plot

In [ ]:
timeline_plot.save(str(FIGURE_DIR / "regime_timeline.png"), scale_factor=2.0)

In [ ]:
profile = regime_profile(context)
t = PrettyTable(["Regime", "Days", "Share %", "Real price", "Reservoir", "Mean ONI", "El Nino %"])
for _, row in profile.iterrows():
    t.add_row([int(row["regime"]), int(row["days"]), f"{row['share_pct']:.1f}", f"{row['mean_real_price']:.1f}",
               f"{row['mean_reservoir']:.3f}", f"{row['mean_oni']:.2f}", f"{row['el_nino_pct']:.1f}"])
print(t)

In [ ]:
metrics = {"model": "conv_vae", "k": K, **summary_metrics(latents, labels, seed=CONFIG["seed"])}
save_regimes(MODEL_DIR, context, labels, probs, metrics)

t = PrettyTable(["Metric", "Value"])
for key, value in metrics.items():
    t.add_row([key, f"{value:.3f}" if isinstance(value, float) else value])
print(t)

In [ ]:
spread = context.groupby("regime")["precio_real_cop_kwh"].describe(percentiles=[0.1, 0.5, 0.9])
spread[["count", "10%", "50%", "90%", "min", "max"]].round(1)